# 3.3 · Benchmark de modelos (regresión y clasificación)

Compara varios modelos sobre **el mismo preprocesado y las mismas particiones** que
`3.1.ModelizaciónRF_vf` (normalización acústica por sexo → drop gender → estandarización
→ SelectKBest(20) → modelo). Dos objetivos:

1. **Sección A — Regresión.** Los modelos sugeridos (Lasso, ElasticNet, SVR, XGBoost)
   junto al Random Forest, en la formulación de regresión del PHQ-8. Sirve para comprobar
   si algún modelo mejora al RF o si **convergen** (evidencia de techo de la modalidad).
2. **Sección B — Clasificación.** Los mismos modelos como **clasificadores** de la tarea
   real (deprimido/no), con balanceo de clases. Métricas estándar de clasificación
   (Accuracy, AUC, F1) — sin RMSE.

Todas las cifras de test se acompañan de **intervalos de confianza bootstrap**, porque con
46 sesiones de test las diferencias entre modelos son en gran parte ruido.

## 0. Setup

In [8]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from sklearn.linear_model import Lasso, ElasticNet, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              GradientBoostingRegressor, GradientBoostingClassifier)

from sklearn.metrics import (mean_squared_error, r2_score, roc_auc_score,
                             f1_score, accuracy_score, precision_recall_fscore_support)

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print('xgboost no disponible (', type(e).__name__, ') -> se omite. Instala con: pip install xgboost')

import warnings
warnings.filterwarnings('ignore')

In [9]:
##################################################
####            Constantes (idénticas al RF)   ####
##################################################
from pathlib import Path
PROJECT_DIR = Path.home().as_posix() + '/Desktop/Master/UNIR_IA_TFE'   # ruta local del proyecto
path_df_features = PROJECT_DIR + '/output/df_acoustic_features_vf.csv'

PHQ_THRESHOLD = 10
RANDOM_STATE  = 42
N_CV_FOLDS    = 5
K_FEATURES    = 20

COLS_EXCLUDE = ['participant_id', 'phq8_binary', 'path_audio',
                'path_transcript', 'split', 'durAudio', 'durSession']
TARGET = 'phq8_score'

ACOUSTIC_PREFIXES = ('f0_', 'energy_', 'mfcc', 'centroid_', 'bandwidth_',
                     'rolloff_', 'zcr_', 'hnr_', 'jitter_', 'shimmer_')

In [10]:
df = pd.read_csv(path_df_features)
COLS_FEATURES = [c for c in df.columns if c not in COLS_EXCLUDE + [TARGET]]
ACOUSTIC_COLS = [c for c in COLS_FEATURES if c.startswith(ACOUSTIC_PREFIXES)]

df_traindev = df[df['split'].isin(['train', 'dev'])].reset_index(drop=True)
df_test     = df[df['split'] == 'test'].reset_index(drop=True)

X_traindev = df_traindev[COLS_FEATURES]
y_traindev = df_traindev[TARGET]
X_test     = df_test[COLS_FEATURES]
y_test     = df_test[TARGET]

y_traindev_bin = (y_traindev >= PHQ_THRESHOLD).astype(int)
y_test_bin     = (y_test     >= PHQ_THRESHOLD).astype(int)

cv_strat  = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv_strat.split(X_traindev, y_traindev_bin))

print(f"Train+Dev: {len(X_traindev)}  ({y_traindev_bin.sum()} deprimidos)")
print(f"Test     : {len(X_test)}  ({y_test_bin.sum()} deprimidos)")

Train+Dev: 140  (43 deprimidos)
Test     : 46  (14 deprimidos)


In [11]:
class GenderZScoreScaler(BaseEstimator, TransformerMixin):
    """Z-score de las columnas acústicas DENTRO de cada grupo de sexo (sin fuga en CV)."""
    def __init__(self, cols, gender_col='gender'):
        self.cols = cols; self.gender_col = gender_col
    def fit(self, X, y=None):
        self.stats_ = {}
        for g, sub in X.groupby(self.gender_col):
            mu = sub[self.cols].mean(); sd = sub[self.cols].std(ddof=0).replace(0, 1.0)
            self.stats_[g] = (mu, sd)
        self.global_ = (X[self.cols].mean(), X[self.cols].std(ddof=0).replace(0, 1.0))
        return self
    def transform(self, X):
        X = X.copy()
        for g, idx in X.groupby(self.gender_col).groups.items():
            mu, sd = self.stats_.get(g, self.global_)
            X.loc[idx, self.cols] = (X.loc[idx, self.cols] - mu) / sd
        return X

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X): return X.drop(columns=[c for c in self.cols if c in X.columns])

def make_pipeline(model, task='reg'):
    """Mismo preprocesado que el RF; SelectKBest con f_regression (reg) o f_classif (clf)."""
    score_func = f_regression if task == 'reg' else f_classif
    return Pipeline([
        ('gender',     GenderZScoreScaler(ACOUSTIC_COLS)),
        ('dropgender', DropColumns(['gender'])),
        ('scaler',     StandardScaler()),
        ('select',     SelectKBest(score_func, k=K_FEATURES)),
        ('model',      model),
    ])

def bootstrap_auc_ci(y_true, y_score, n_boot=2000, seed=RANDOM_STATE):
    """IC 95% percentil del AUC por bootstrap sobre las muestras de test."""
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    rng = np.random.RandomState(seed); n = len(y_true); aucs = []
    for _ in range(n_boot):
        s = rng.randint(0, n, n)
        if len(np.unique(y_true[s])) < 2: continue
        aucs.append(roc_auc_score(y_true[s], y_score[s]))
    return np.percentile(aucs, [2.5, 97.5])

## A. Benchmark en regresión (PHQ-8 continuo)

Modelos sugeridos por revisión + RF, en la formulación de regresión. Para cada uno:
métricas de regresión (RMSE, R²) en CV y test, y las de clasificación *derivadas*
(AUC sobre la predicción continua; F1 con el umbral que lo maximiza sobre las
predicciones out-of-fold de train+dev).

In [12]:
reg_models = {
    'Lasso':        Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=10000),
    'ElasticNet':   ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000),
    'SVR (RBF)':    SVR(kernel='rbf', C=1.0, gamma='scale'),
    'RandomForest': RandomForestRegressor(n_estimators=500, max_depth=10, min_samples_leaf=2,
                                          min_samples_split=24, max_features='log2',
                                          random_state=RANDOM_STATE, n_jobs=-1),
    'GradBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}
if HAS_XGB:
    reg_models['XGBoost'] = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.03,
                                         subsample=0.8, colsample_bytree=0.8,
                                         random_state=RANDOM_STATE, n_jobs=-1)

def best_f1_threshold(y_bin, y_score):
    ths = np.linspace(np.min(y_score), np.max(y_score), 200)
    f1s = [f1_score(y_bin, (y_score >= t).astype(int), zero_division=0) for t in ths]
    return ths[int(np.argmax(f1s))]

rows = []
for name, mdl in reg_models.items():
    pipe = make_pipeline(mdl, task='reg')
    # OOF continuo sobre train+dev (sin fuga)
    oof = cross_val_predict(clone(pipe), X_traindev, y_traindev, cv=cv_splits)
    thr = best_f1_threshold(y_traindev_bin.values, oof)
    # test
    pipe.fit(X_traindev, y_traindev)
    pred = pipe.predict(X_test)
    lo, hi = bootstrap_auc_ci(y_test_bin.values, pred)
    rows.append({
        'Modelo': name,
        'RMSE_cv':  np.sqrt(mean_squared_error(y_traindev, oof)),
        'R2_cv':    r2_score(y_traindev, oof),
        'RMSE_test': np.sqrt(mean_squared_error(y_test, pred)),
        'R2_test':  r2_score(y_test, pred),
        'AUC_test': roc_auc_score(y_test_bin, pred),
        'AUC_IC95': f"[{lo:.2f}, {hi:.2f}]",
        'F1_test':  f1_score(y_test_bin, (pred >= thr).astype(int), zero_division=0),
    })

reg_results = pd.DataFrame(rows).set_index('Modelo').round(3)
print("── Benchmark REGRESIÓN ──")
reg_results

── Benchmark REGRESIÓN ──


,RMSE_cv,R2_cv,RMSE_test,R2_test,AUC_test,AUC_IC95,F1_test
Modelo,,,,,,,
Lasso,5.553,0.073,6.377,0.011,0.658,"[0.46, 0.84]",0.526
ElasticNet,5.527,0.082,6.371,0.012,0.665,"[0.47, 0.84]",0.471
SVR (RBF),5.754,0.005,6.350,0.019,0.667,"[0.49, 0.85]",0.500
RandomForest,5.599,0.058,6.115,0.090,0.743,"[0.57, 0.89]",0.509
GradBoosting,5.936,-0.059,6.572,-0.051,0.667,"[0.49, 0.83]",0.491
XGBoost,5.852,-0.029,6.570,-0.050,0.663,"[0.48, 0.84]",0.468


## B. Benchmark en clasificación (deprimido / no deprimido)

Los mismos modelos como **clasificadores** de la tarea real, con **balanceo de clases**
(`class_weight='balanced'` / `scale_pos_weight`). Métricas estándar de clasificación,
**sin RMSE**. El AUC se calcula sobre la probabilidad predicha; Accuracy y F1 con el
umbral de probabilidad 0.5.

In [13]:
neg, pos = (y_traindev_bin == 0).sum(), (y_traindev_bin == 1).sum()
spw = neg / pos   # scale_pos_weight para XGBoost

clf_models = {
    'LogReg L2':      LogisticRegression(class_weight='balanced', max_iter=5000),
    'LogReg ElNet':   LogisticRegression(penalty='elasticnet', l1_ratio=0.5, solver='saga',
                                         class_weight='balanced', max_iter=10000),
    'SVM (RBF)':      SVC(kernel='rbf', probability=True, class_weight='balanced',
                          random_state=RANDOM_STATE),
    'RandomForest':   RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_leaf=2,
                                             min_samples_split=24, max_features='log2',
                                             class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'GradBoosting':   GradientBoostingClassifier(random_state=RANDOM_STATE),
}
if HAS_XGB:
    clf_models['XGBoost'] = XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.03,
                                          subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                                          eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)

rows = []
for name, mdl in clf_models.items():
    pipe = make_pipeline(mdl, task='clf')
    # OOF probabilidad sobre train+dev
    oof_p = cross_val_predict(clone(pipe), X_traindev, y_traindev_bin, cv=cv_splits,
                              method='predict_proba')[:, 1]
    auc_cv = roc_auc_score(y_traindev_bin, oof_p)
    # test
    pipe.fit(X_traindev, y_traindev_bin)
    p_test = pipe.predict_proba(X_test)[:, 1]
    yhat   = (p_test >= 0.5).astype(int)
    lo, hi = bootstrap_auc_ci(y_test_bin.values, p_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test_bin, yhat, average='binary', zero_division=0)
    rows.append({
        'Modelo': name,
        'AUC_cv':   auc_cv,
        'AUC_test': roc_auc_score(y_test_bin, p_test),
        'AUC_IC95': f"[{lo:.2f}, {hi:.2f}]",
        'Acc_test': accuracy_score(y_test_bin, yhat),
        'Prec_test': prec,
        'Rec_test':  rec,
        'F1_test':   f1,
    })

clf_results = pd.DataFrame(rows).set_index('Modelo').round(3)
print(f"── Benchmark CLASIFICACIÓN ──  (baseline accuracy = {max(neg,pos)/(neg+pos):.3f})")
clf_results

── Benchmark CLASIFICACIÓN ──  (baseline accuracy = 0.693)


,AUC_cv,AUC_test,AUC_IC95,Acc_test,Prec_test,Rec_test,F1_test
Modelo,,,,,,,
LogReg L2,0.642,0.585,"[0.37, 0.79]",0.587,0.381,0.571,0.457
LogReg ElNet,0.634,0.583,"[0.37, 0.79]",0.587,0.381,0.571,0.457
SVM (RBF),0.428,0.531,"[0.33, 0.73]",0.696,0.000,0.000,0.000
RandomForest,0.523,0.654,"[0.46, 0.84]",0.696,0.500,0.571,0.533
GradBoosting,0.560,0.578,"[0.39, 0.76]",0.696,0.500,0.357,0.417
XGBoost,0.566,0.563,"[0.36, 0.75]",0.630,0.385,0.357,0.370


## C. Lectura

- **Baseline de accuracy** (predecir siempre la clase mayoritaria) = proporción de no deprimidos.
  Cualquier Accuracy debe compararse contra ese valor, no contra 0.5.
- Si los modelos **convergen** (AUC solapados, IC que se pisan), es evidencia de que el techo
  está en la **modalidad y el tamaño muestral**, no en el algoritmo — que es la tesis del trabajo.
- El **IC bootstrap** muestra que, con n_test=46, las diferencias entre modelos son en su
  mayoría ruido: no se debe elegir "el mejor" por su punto estimado en test.